In [5]:
import os

# Set Java 17 as the primary Java for this process
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = "/usr/lib/jvm/java-17-openjdk-amd64/bin:" + os.environ["PATH"]

import pyspark
from pyspark.sql import SparkSession

# Initialize the session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('yellow_taxi') \
    .getOrCreate()

# Verify the version again
print("Java Version in Spark:", spark._jvm.java.lang.System.getProperty("java.version"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 21:22:48 WARN Utils: Your hostname, codespaces-14c820, resolves to a loopback address: 127.0.0.1; using 10.0.1.22 instead (on interface eth0)
26/03/10 21:22:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 21:22:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/10 21:22:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Java Version in Spark: 17.0.18


In [6]:
df = spark.read.parquet('/workspaces/dataengineering-zoomcamp/06-batch-processing/data/raw/yellow/yellow_tripdata_2025-11.parquet')

In [7]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [8]:
df_repartitioned = df.repartition(4)

In [9]:
df_repartitioned.write.mode('overwrite').parquet('/workspaces/dataengineering-zoomcamp/06-batch-processing/data/pq/yellow_taxi_data_partitioned')

In [10]:
df_repartitioned.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

In [9]:
from pyspark.sql import functions as F

count = df_repartitioned.filter(F.to_date('tpep_pickup_datetime') == '2025-11-15').count()
print(f"Number of trips on 2025-11-15: {count}")

Number of trips on 2025-11-15: 162604


In [10]:
from pyspark.sql import functions as F

# 1. Calculate duration in seconds
df_duration = df_repartitioned.withColumn(
    "duration_seconds",
    F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
)

# 2. Extract the maximum value
longest_trip_seconds = df_duration.select(F.max("duration_seconds")).collect()[0][0]

# 3. Convert to hours for analysis
longest_trip_hours = longest_trip_seconds / 3600
print(f"Longest trip: {longest_trip_hours:.2f} hours")

Longest trip: 90.65 hours


In [9]:
# 1. Load the CSV into a DataFrame
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/workspaces/dataengineering-zoomcamp/06-batch-processing/data/raw/taxi_zone_lookup.csv")

# 2. Register as a Temporary View
df_zones.createOrReplaceTempView("zones")

In [10]:
# Query the view using SQL
spark.sql("""
    SELECT * FROM zones 
    LIMIT 5
""").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+



In [11]:
df_repartitioned.createOrReplaceTempView("trips")

In [12]:
# Identify the Least frequent pickup location zone
result = spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(1) AS trip_count
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY COUNT(1) ASC
    LIMIT 5
""")

result.show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
|Eltingville/Annad...|         1|
|       Arden Heights|         1|
|       Port Richmond|         3|
| Green-Wood Cemetery|         4|
+--------------------+----------+

